# Lab 22-01 · Power Conversion Principles

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import power as pw


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The leverage of efficiency

*Predict* how much heat per kilowatt delivered a supply produces at 90, 96 and 99 % efficiency.

In [2]:
for eta in (0.90, 0.96, 0.965, 0.98, 0.99):
    print(f"eta {eta*100:5.1f} %: loss {pw.loss_from_efficiency(1000, eta):6.2f} W per kW delivered")
eta = np.linspace(0.80, 0.995, 400)
plt.figure(); plt.plot(eta * 100, pw.loss_from_efficiency(1000, eta)); plt.yscale("log")
plt.xlabel("efficiency (%)"); plt.ylabel("loss per kW delivered (W)"); plt.show()

eta  90.0 %: loss 111.11 W per kW delivered
eta  96.0 %: loss  41.67 W per kW delivered
eta  96.5 %: loss  36.27 W per kW delivered
eta  98.0 %: loss  20.41 W per kW delivered
eta  99.0 %: loss  10.10 W per kW delivered


## Part 2 · A linear regulator at heavy and light load

*Predict* the efficiency of a 3.6 V to 1.8 V linear regulator with 25 nA quiescent current at 10 mA and at 1 µA, then run `ldo.cir`.

In [3]:
for i in (10e-3, 1e-6):
    print(f"I_out {i*1e6:8.1f} uA: eta {pw.ldo_efficiency(3.6, 1.8, i, 25e-9)*100:.2f} %")
s = ngspice("ldo.cir")
print({k: round(s[k], 5) for k in ("eta10m", "eta1u", "pl10m")})
print(f"buck battery current at 1 uA (80 %): {pw.buck_input_current(3.6, 1.8, 1e-6, 0.8)*1e6:.3f} uA")
i = np.logspace(-9, -1, 400)
plt.figure()
for iq in (25e-9, 1e-6, 50e-6):
    plt.semilogx(i, pw.ldo_efficiency(3.6, 1.8, i, iq) * 100, label=f"I_Q = {iq*1e9:g} nA")
plt.axhline(50, ls=":", color="0.5"); plt.xlabel("load current (A)"); plt.ylabel("efficiency (%)"); plt.legend(); plt.show()

I_out  10000.0 uA: eta 50.00 %
I_out      1.0 uA: eta 48.78 %
{'eta10m': 0.50001, 'eta1u': 0.48781, 'pl10m': 0.018}
buck battery current at 1 uA (80 %): 0.625 uA


## Part 3 · The price of charging a capacitor

*Predict* the energy lost charging 10 µF to 5 V through 1 Ω, through 100 Ω and in four steps, then integrate numerically and run `cap_charge.cir`.

In [4]:
from scipy.integrate import solve_ivp
for r in (0.1, 1.0, 100.0):
    c, vs = 10e-6, 5.0
    sol = solve_ivp(lambda t, y: [(vs - y[0]) / (r * c), (vs - y[0]) ** 2 / r], (0, 40 * r * c), [0, 0], rtol=1e-10, atol=1e-15)
    print(f"R = {r:6.1f} ohm: lost {sol.y[1, -1]*1e6:.3f} uJ (formula {pw.charge_loss(c, vs)*1e6:.3f} uJ)")
print(f"four steps: {pw.charge_loss(10e-6, 5.0, 4)*1e6:.3f} uJ; sharing 5.0 V and 4.8 V on 1 uF each: {pw.share_loss(1e-6, 5.0, 1e-6, 4.8)*1e9:.2f} nJ")
s = ngspice("cap_charge.cir")
print({k: s[k] for k in ("e1", "e100", "estep")})

R =    0.1 ohm: lost 125.000 uJ (formula 125.000 uJ)
R =    1.0 ohm: lost 125.000 uJ (formula 125.000 uJ)


R =  100.0 ohm: lost 125.000 uJ (formula 125.000 uJ)
four steps: 31.250 uJ; sharing 5.0 V and 4.8 V on 1 uF each: 10.00 nJ


{'e1': 0.000125003, 'e100': 0.000124991, 'estep': 3.125e-05}


## Part 4 · Switching plus filtering

*Predict* the average output and the peak-to-peak ripple of the 12 V to 3.3 V buck of Example 22-01.4, then compute its exact periodic steady state, run `buck.cir`, and check what happens when $f_s$ doubles while $L$ and $C$ halve.

In [5]:
D = 3.3 / 12
t, il, vo = pw.buck_pss(12.0, D, 500e3, 10e-6, 1e-6, 3.3)
print(f"average {np.trapezoid(vo, t)/t[-1]:.4f} V, ripple {(vo.max()-vo.min())*1e3:.2f} mV p-p, inductor ripple {il.max()-il.min():.4f} A p-p")
print(f"small-ripple formula for the inductor ripple: {(12-3.3)*D/500e3/10e-6:.4f} A")
a1 = pw.switch_node_harmonic(12.0, D, 1)
f0 = 1 / (2 * math.pi * math.sqrt(10e-12))
print(f"fundamental {a1:.3f} V, first-harmonic ripple estimate {2*a1/abs(1-(500e3/f0)**2)*1e3:.1f} mV")
t2, il2, vo2 = pw.buck_pss(12.0, D, 1e6, 5e-6, 0.5e-6, 3.3)
print(f"1 MHz, 5 uH, 0.5 uF: ripple {(vo2.max()-vo2.min())*1e3:.2f} mV")
s = ngspice("buck.cir")
print({k: s[k] for k in ("vavg", "vpp", "iin")})
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(t * 1e6, il); ax[0].set_ylabel("i_L (A)")
ax[1].plot(t * 1e6, vo); ax[1].set_ylabel("v_out (V)"); ax[1].set_xlabel("time in period (us)"); plt.show()

average 3.3000 V, ripple 120.33 mV p-p, inductor ripple 0.4817 A p-p
small-ripple formula for the inductor ripple: 0.4785 A
fundamental 5.809 V, first-harmonic ripple estimate 118.9 mV
1 MHz, 5 uH, 0.5 uF: ripple 120.33 mV


{'vavg': 3.299413, 'vpp': 0.120535, 'iin': 0.2750326}


## Part 5 · The loss model, light load and phases

*Predict* where the converter of Example 22-01.5 peaks, and the loads at which a second and third phase should turn on.

In [6]:
p0, a, r, vout = 0.150, 0.030, 0.003, 1.0
print(f"peak at {pw.peak_efficiency_current(p0, r):.3f} A, eta {pw.model_efficiency(pw.peak_efficiency_current(p0, r), vout, p0, a, r)*100:.2f} %")
for n in (1, 2, 3):
    print(f"add phase {n+1} above {pw.shed_current(n, p0, r):.2f} A")
i = np.logspace(-2, np.log10(30), 400)
plt.figure()
plt.semilogx(i, pw.model_efficiency(i, vout, p0, a, r) * 100, label="one phase, fixed frequency")
plt.semilogx(i, pw.model_efficiency(i, vout, p0 / 10, a, r) * 100, label="one phase, PFM (P0 / 10)")
plt.semilogx(i, vout * i / (vout * i + pw.phased_loss(i, 2, p0, a, r)) * 100, label="two phases")
plt.ylim(0, 100); plt.xlabel("load (A)"); plt.ylabel("efficiency (%)"); plt.legend(); plt.show()

peak at 7.071 A, eta 93.25 %
add phase 2 above 10.00 A
add phase 3 above 17.32 A
add phase 4 above 24.49 A


## Part 6 · Density, the thermal limit and hold-up

*Predict* the density of the 18 kW and 5.5 kW supplies of Example 22-01.6, and the hold-up capacitance of Example 22-01.7.

In [7]:
for p, dims in ((18e3, (104, 710, 40)), (5.5e3, (640, 73.5, 40))):
    rho, vol = pw.density_w_per_in3(p, dims)
    print(f"{p/1e3:.1f} kW in {vol:.1f} in^3: {rho:.1f} W/in^3")
for eta in (0.95, 0.975, 0.99):
    print(f"eta {eta*100:.1f} %: thermal density limit at 3 W/in^3 = {pw.thermal_density_limit(3, eta):.0f} W/in^3")
pb = 5500 / 0.985
for v2 in (320, 300):
    print(f"hold-up to {v2} V: C = {pw.holdup_capacitance(pb, 0.020, 400, v2)*1e3:.2f} mF")
losses, p_in = pw.chain_losses(1000, [0.97, 0.975, 0.977, 0.90])
print("chain losses (W):", [round(x, 1) for x in losses], f"input {p_in:.1f} W")

18.0 kW in 180.2 in^3: 99.9 W/in^3
5.5 kW in 114.8 in^3: 47.9 W/in^3
eta 95.0 %: thermal density limit at 3 W/in^3 = 57 W/in^3
eta 97.5 %: thermal density limit at 3 W/in^3 = 117 W/in^3
eta 99.0 %: thermal density limit at 3 W/in^3 = 297 W/in^3
hold-up to 320 V: C = 3.88 mF
hold-up to 300 V: C = 3.19 mF
chain losses (W): [36.1, 29.2, 26.2, 111.1] input 1202.5 W


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.